## Model Interpretation only (due to cacheing issue in Databricks Free Tier)

This notebook focuses on:
- model interpretation (what features drive predictions)
- report-ready summary tables

#### Load up stored data

In [0]:
from pyspark.sql import functions as SQL_FUNCTIONS
from pyspark.ml.functions import vector_to_array

FINAL_SELECTION_TABLE = "workspace.bda_taxi.model_comparison_final"
BEST_PRED_TABLE = "workspace.bda_taxi.model_preds_best"
THRESHOLD_TABLE = "workspace.bda_taxi.model_threshold_metrics"

final_selection = spark.table(FINAL_SELECTION_TABLE)
best_preds = spark.table(BEST_PRED_TABLE)

display(final_selection)
print("Best predictions rows:", best_preds.count())
display(best_preds.limit(5))

## Model interpretation (refitting best model only)

Refitting the best model that was stored in a table in previous notebook.
Depending on the model that it is:
- Logistic Regression: coefficients
- Random Forest: feature importances

##### Reuse the pipeline builders and feature lists from shared notebook 04

In [0]:
%run ./04_model_training_and_evaluation_shared



In [0]:
# Model interpretation (Random Forest) — feature importance by original columns
# This avoids one-hot feature name mapping and produces an interpretable importance per input column.

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

TRAIN_TABLE = "workspace.bda_taxi.model_train_set"
TEST_TABLE = "workspace.bda_taxi.model_test_set"

train_df = spark.table(TRAIN_TABLE)
test_df = spark.table(TEST_TABLE)

label_column = "tipped"

# Use the same feature lists from the training notebook
numeric_cols = numeric_features
categorical_cols = categorical_features

# Index categoricals (no OneHot) to keep feature count small and interpretability high
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

feature_cols = numeric_cols + [f"{c}_idx" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_column,
    numTrees=30,
    maxDepth=8,
    subsamplingRate=0.7,
    featureSubsetStrategy="sqrt",
    # MUST be >= max categorical cardinality (zones ~259)
    maxBins=512,
    seed=42
)

pipeline = Pipeline(stages=indexers + [assembler, rf])
model = pipeline.fit(train_df)

rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()

# Map importances back to original columns
importance_pairs = list(zip(feature_cols, [float(x) for x in importances]))
importance_pairs_sorted = sorted(importance_pairs, key=lambda x: x[1], reverse=True)

importance_df = spark.createDataFrame(importance_pairs_sorted, ["feature", "importance"])
display(importance_df.limit(30))